# ⛽ Mini Project 3B — Pekan 3: Benchmark & Refactoring
**Gas Optimization Framework untuk Smart Contract Solidity**

Jalankan sel dari atas ke bawah secara berurutan.

| Sel | Isi |
|---|---|
| 1 | Setup & install Slither |
| 2 | Hardhat gas benchmark (6 anti-pattern) |
| 3 | Demo refactorer pada MultiSigWallet |
| 4 | Verifikasi refactored code masih compile |
| 5 | Slither comparison pada 10 kontrak |
| 6 | Head-to-head: detector kita vs Slither |
| 7 | Gabungkan semua hasil ke tabel eksperimen |
| 8 | Checklist akhir Pekan 3 |

---
## ⚙️ Sel 1 — Setup & Install Slither

In [1]:
import sys, json, subprocess, re
from pathlib import Path
from collections import defaultdict

ROOT        = Path().resolve().parent
HARDHAT_DIR = ROOT / 'hardhat_project'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ast_parser   import generate_ast
from src.detectors    import ALL_DETECTORS
from src.gas_estimator import measure_all, print_results
from src.refactorer   import apply_all_refactors, diff_summary

# Load dari contracts_selection.json (dataset aktif)
SELECTION_FILE = ROOT / 'contracts_selection.json'
if not SELECTION_FILE.exists():
    SELECTION_FILE = ROOT / 'contracts_metadata_expanded.json'
    print(f'⚠️  Fallback ke {SELECTION_FILE.name}')

with open(SELECTION_FILE) as f:
    metadata = json.load(f)
valid = [m for m in metadata if m.get('compile_ok', True)]

def run(cmd, cwd=None, timeout=120):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                       cwd=cwd, timeout=timeout)
    out = r.stdout.strip(); err = r.stderr.strip()
    if out: print(out)
    if r.returncode != 0 and err: print('STDERR:', err[:300])
    return r

try:
    import slither
    print('✅ Slither sudah terinstall')
except ImportError:
    print('⏳ Installing slither-analyzer...')
    run(f'{sys.executable} -m pip install slither-analyzer --quiet')
    try:
        import slither
        print('✅ Slither berhasil diinstall')
    except ImportError:
        print('⚠️  Slither gagal install — Sel 5 & 6 akan di-skip')

print(f'\n✅ Setup selesai | sumber: {SELECTION_FILE.name} | valid: {len(valid)}')

✅ Slither sudah terinstall

✅ Setup selesai | sumber: contracts_selection.json | valid: 74


---
## ⛽ Sel 2 — Hardhat Gas Benchmark (6 Anti-Pattern)

In [2]:
print('⏳ Menulis benchmark contract + menjalankan Hardhat...\n')

bench_results = measure_all()

print('=== HASIL BENCHMARK GAS ===')
print(f'{"Anti-Pattern":<28} {"Boros":>9} {"Hemat":>9} {"Selisih":>9} {"Hemat%":>7}')
print('─' * 67)
for r in bench_results:
    sign = '+' if r['diff_gas'] > 0 else ''
    print(
        f'{r["pattern"]:<28} '
        f'{r["boros_gas"]:>9,} '
        f'{r["hemat_gas"]:>9,} '
        f'{sign}{r["diff_gas"]:>8,} '
        f'{r["pct_save"]:>6.1f}%'
    )
print('─' * 67)

# Simpan ke results/
bench_path = RESULTS_DIR / 'pekan3_gas_benchmark.json'
with open(bench_path, 'w') as f:
    json.dump(bench_results, f, indent=2)
print(f'\n💾 Disimpan: {bench_path}')

print('\n📋 Interpretasi:')
for r in bench_results:
    if r['pct_save'] > 0:
        print(f'  • {r["pattern"]}: hemat {r["pct_save"]}% ({r["diff_gas"]:,} gas)')
    else:
        print(f'  • {r["pattern"]}: tidak ada penghematan (atau overhead kecil)')

⏳ Menulis benchmark contract + menjalankan Hardhat...

=== HASIL BENCHMARK GAS ===
Anti-Pattern                     Boros     Hemat   Selisih  Hemat%
───────────────────────────────────────────────────────────────────
redundant_sload                 24,208    24,022 +     186    0.8%
unoptimized_loop                51,187    50,156 +   1,031    2.0%
string_vs_bytes32               24,540    23,590 +     950    3.9%
public_vs_external              52,544    49,871 +   2,673    5.1%
unchecked_arithmetic            59,105    47,060 +  12,045   20.4%
dead_code                      123,985   123,985        0    0.0%
───────────────────────────────────────────────────────────────────

💾 Disimpan: C:\Users\Ridho\project\KBJ\gas_optimization\results\pekan3_gas_benchmark.json

📋 Interpretasi:
  • redundant_sload: hemat 0.77% (186 gas)
  • unoptimized_loop: hemat 2.01% (1,031 gas)
  • string_vs_bytes32: hemat 3.87% (950 gas)
  • public_vs_external: hemat 5.09% (2,673 gas)
  • unchecked_arithmeti

---
## 🔧 Sel 3 — Demo Refactorer pada MultiSigWallet
MultiSigWallet memiliki 5 `unoptimized_loop` findings — cocok untuk demo refactoring.

In [3]:
DEMO_CONTRACT = 'Utility_03_MultiSigWallet.sol'

demo_meta = next(
    (m for m in valid if Path(m['file']).name == DEMO_CONTRACT),
    valid[0]
)
print(f'📄 Kontrak demo: {demo_meta["nama"]} ({demo_meta["domain"]})')

# ── Jalankan semua detektor
demo_ast = generate_ast(demo_meta['file'])
findings_by_type = {}
print('\nFindings sebelum refactoring:')
for name, detect in ALL_DETECTORS:
    f = detect(demo_ast)
    findings_by_type[name] = f
    if f:
        print(f'  {name}: {len(f)} temuan')
        for item in f[:2]:
            print(f'    baris {item["line"]}: {item["description"][:70]}...')

# ── Terapkan refactoring
original_src  = Path(demo_meta['file']).read_text(encoding='utf-8')
refactored_src, summary = apply_all_refactors(demo_meta['file'], findings_by_type)

delta = diff_summary(original_src, refactored_src)
print(f'\n🔧 Refactoring diterapkan:')
for k, v in summary.items():
    print(f'  • {k}: {v} patch')
print(f'  Baris diubah  : {delta["lines_changed"]}')
print(f'  Baris ditambah: {delta["lines_added"]}')

# ── Simpan hasil
refactored_path = RESULTS_DIR / f'refactored_{demo_meta["nama"]}.sol'
refactored_path.write_text(refactored_src, encoding='utf-8')
print(f'\n💾 Refactored source: {refactored_path}')

# ── Preview diff pada beberapa baris
print('\n📋 Preview perubahan (baris yang berbeda):')
orig_lines = original_src.splitlines()
refc_lines = refactored_src.splitlines()
shown = 0
for i, (a, b) in enumerate(zip(orig_lines, refc_lines)):
    if a != b and shown < 8:
        print(f'  L{i+1:4d} - {a.strip()[:70]}')
        print(f'       + {b.strip()[:70]}')
        shown += 1
if len(refc_lines) > len(orig_lines):
    for line in refc_lines[len(orig_lines):][:4]:
        print(f'  +new  + {line.strip()[:70]}')

📄 Kontrak demo: MultiSigWallet (Utility)

Findings sebelum refactoring:
  redundant_sload: 11 temuan
    baris None: Redundant SLOAD: 'isOwner' dibaca 2x di fungsi 'MultiSigWallet'. Cache...
    baris None: Redundant SLOAD: 'owners' dibaca 8x di fungsi 'removeOwner'. Cache ke ...
  unoptimized_loop: 5 temuan
    baris None: Unoptimized Loop: 'owners.length' dibaca dari storage di setiap iteras...
    baris None: Unoptimized Loop: 'owners.length' dibaca dari storage di setiap iteras...
  public_vs_external: 10 temuan
    baris None: Public vs External: fungsi 'addOwner' dideklarasikan `public` tapi tid...
    baris None: Public vs External: fungsi 'removeOwner' dideklarasikan `public` tapi ...

🔧 Refactoring diterapkan:
  • public_vs_external: 10 patch
  • unoptimized_loop: 5 patch
  • redundant_sload: 11 patch
  Baris diubah  : 317
  Baris ditambah: 5

💾 Refactored source: C:\Users\Ridho\project\KBJ\gas_optimization\results\refactored_MultiSigWallet.sol

📋 Preview perubahan (baris yang

---
## ✅ Sel 4 — Verifikasi Refactored Code Masih Compile

In [4]:
import solcx
from src.ast_parser import detect_solc_ver, dedup_pragma

def try_compile(source, label=''):
    ver = detect_solc_ver(source)
    try:
        from packaging.version import Version as V
        if V(ver) < V('0.4.11'): ver = '0.4.26'
    except Exception:
        pass
    installed = [str(v) for v in solcx.get_installed_solc_versions()]
    if ver not in installed: solcx.install_solc(ver)
    try:
        src_clean = dedup_pragma(source)
        out = solcx.compile_source(src_clean, output_values=['abi'],
                                   solc_version=ver, optimize=False)
        return True, None
    except Exception as e:
        return False, str(e)[:120]

# ── Test original
ok_orig, err_orig = try_compile(original_src, 'original')
print(f'Original   : {"✅ compile OK" if ok_orig else "❌ " + err_orig}')

# ── Test refactored
ok_refc, err_refc = try_compile(refactored_src, 'refactored')
print(f'Refactored : {"✅ compile OK" if ok_refc else "❌ " + err_refc}')

# ── Re-run detectors on refactored version
if ok_refc:
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.sol', mode='w',
                                     encoding='utf-8', delete=False) as tmp:
        tmp.write(refactored_src)
        tmp_path = tmp.name
    refc_ast = generate_ast(tmp_path)
    Path(tmp_path).unlink(missing_ok=True)

    if refc_ast:
        print('\nFindings SETELAH refactoring:')
        total_before = sum(len(findings_by_type.get(n, [])) for n, _ in ALL_DETECTORS)
        total_after  = 0
        for name, detect in ALL_DETECTORS:
            f_after  = detect(refc_ast)
            f_before = findings_by_type.get(name, [])
            total_after += len(f_after)
            if f_before or f_after:
                delta_str = f'-{len(f_before)-len(f_after)}' if len(f_before) > len(f_after) else ''
                print(f'  {name}: {len(f_before)} → {len(f_after)} {delta_str}')
        print(f'\nTotal: {total_before} → {total_after} findings '
              f'(berkurang {total_before-total_after})')

Original   : ✅ compile OK
Refactored : ✅ compile OK

Findings SETELAH refactoring:
  redundant_sload: 11 → 11 
  unoptimized_loop: 5 → 0 -5
  public_vs_external: 10 → 10 

Total: 26 → 21 findings (berkurang 5)


---
## 🔍 Sel 5 — Slither: Analisis 10 Kontrak

In [5]:
import shutil, tempfile

slither_exe = shutil.which('slither')
slither_ok  = slither_exe is not None

if not slither_ok:
    print('⚠️  Slither tidak tersedia — lewati Sel 5 & 6')
    slither_results = {}
else:
    print(f'🔍 Slither ditemukan: {slither_exe}')
    print('   Menjalankan analisis pada 10 kontrak...\n')

    sample = [m for m in valid if m['complexity'] in ('Medium', 'Simple')][:10]
    if len(sample) < 10:
        sample = valid[:10]

    slither_results = {}
    GAS_DETECTORS = [
        'costly-loop', 'dead-code', 'unused-return',
        'cache-array-length', 'storage-array', 'redundant-statements'
    ]

    for m in sample:
        sol_file = m['file']
        print(f'  Analisis: {m["nama"]}')
        tmp_json = None
        detectors_found = set()
        try:
            with tempfile.NamedTemporaryFile(suffix='.json', delete=False) as tmp:
                tmp_json = tmp.name
            r = subprocess.run(
                [slither_exe, sol_file, '--json', tmp_json, '--disable-color'],
                capture_output=True, text=True, timeout=60
            )
            try:
                with open(tmp_json, encoding='utf-8') as f:
                    data = json.load(f)
                for det in data.get('results', {}).get('detectors', []):
                    detectors_found.add(det.get('check', ''))
            except Exception:
                pass
        except subprocess.TimeoutExpired:
            print('    → timeout')
        except Exception as e:
            print(f'    → error: {e}')
        finally:
            if tmp_json:
                Path(tmp_json).unlink(missing_ok=True)

        gas_hits = [d for d in detectors_found if d in GAS_DETECTORS]
        slither_results[m['nama']] = {
            'all_detectors': list(detectors_found),
            'gas_detectors': gas_hits,
        }
        print(f'    → {len(detectors_found)} total, {len(gas_hits)} gas-related')

    slither_path = RESULTS_DIR / 'pekan3_slither_results.json'
    with open(slither_path, 'w') as f:
        json.dump(slither_results, f, indent=2)
    print(f'\n💾 Disimpan: {slither_path}')


🔍 Slither ditemukan: C:\Users\Ridho\anaconda3\envs\gas_opt\Scripts\slither.EXE
   Menjalankan analisis pada 10 kontrak...

  Analisis: AppProxyUpgradeable
    → 0 total, 0 gas-related
  Analisis: Dai
    → 0 total, 0 gas-related
  Analisis: DaiJoin
    → 0 total, 0 gas-related
  Analisis: ETHJoin
    → 0 total, 0 gas-related
  Analisis: GemJoin
    → 0 total, 0 gas-related
  Analisis: KyberNetworkProxy
    → 0 total, 0 gas-related
  Analisis: Spotter
    → 0 total, 0 gas-related
  Analisis: UniswapV2Factory
    → 0 total, 0 gas-related
  Analisis: UniswapV2Pair
    → 0 total, 0 gas-related
  Analisis: Vat
    → 0 total, 0 gas-related

💾 Disimpan: C:\Users\Ridho\project\KBJ\gas_optimization\results\pekan3_slither_results.json


---
## 📊 Sel 6 — Head-to-Head: Detektor Kita vs Slither

In [6]:
# Load Pekan 2 results
with open(RESULTS_DIR / 'pekan2_detector_results.json') as f:
    p2_results = json.load(f)
p2_by_name = {r['nama']: r for r in p2_results}

DETECTOR_NAMES = [name for name, _ in ALL_DETECTORS]

if not slither_results:
    print('⚠️  Slither tidak dijalankan — menampilkan statistik Pekan 2 saja')
    print()
    print(f'{"Kontrak":<35} {"Compile":>8}', '  '.join(f'{n[:5]:>6}' for n in DETECTOR_NAMES))
    print('─' * 90)
    for r in p2_results[:15]:
        ok = '✅' if r['compile_ok'] else '❌'
        vals = '  '.join(f'{r.get(d, 0):>6}' for d in DETECTOR_NAMES)
        print(f'{r["nama"]:<35} {ok:>8}  {vals}')
else:
    print(f'{"Kontrak":<30} {"Kita":>6} {"Slither":>8}  Match?')
    print('─' * 60)
    match_count = 0
    for nama, sdata in slither_results.items():
        our_total = sum(
            p2_by_name.get(nama, {}).get(d, 0)
            for d in DETECTOR_NAMES
        ) if nama in p2_by_name else 0
        slither_gas_n = len(sdata['gas_detectors'])
        both_find = our_total > 0 and slither_gas_n > 0
        match     = '✅' if both_find else ('⚠️' if our_total == 0 and slither_gas_n == 0 else '─')
        if both_find: match_count += 1
        print(f'{nama:<30} {our_total:>6} {slither_gas_n:>8}  {match}  {sdata["gas_detectors"]}')
    print(f'\nBoth tools found issues: {match_count}/{len(slither_results)} kontrak')

Kontrak                          Kita  Slither  Match?
────────────────────────────────────────────────────────────
AppProxyUpgradeable                11        0  ─  []
Dai                                10        0  ─  []
DaiJoin                             2        0  ─  []
ETHJoin                             2        0  ─  []
GemJoin                             1        0  ─  []
KyberNetworkProxy                  68        0  ─  []
Spotter                             0        0  ⚠️  []
UniswapV2Factory                   20        0  ─  []
UniswapV2Pair                      14        0  ─  []
Vat                                24        0  ─  []

Both tools found issues: 0/10 kontrak


---
## 📋 Sel 7 — Tabel Eksperimen Lengkap

In [7]:
# ── Tabel 4.4a: Gas Savings per Anti-Pattern (dari Hardhat benchmark)
print('=== TABEL 4.4a: GAS SAVINGS PER ANTI-PATTERN ===')
print(f'{"No":>3} {"Anti-Pattern":<28} {"Gas Boros":>10} {"Gas Hemat":>10} {"Selisih":>9} {"% Hemat":>8}')
print('─' * 75)
for i, r in enumerate(bench_results, 1):
    print(
        f'{i:>3}. {r["pattern"]:<28} '
        f'{r["boros_gas"]:>10,} '
        f'{r["hemat_gas"]:>10,} '
        f'{r["diff_gas"]:>9,} '
        f'{r["pct_save"]:>7.1f}%'
    )

# ── Tabel 4.4b: Distribusi Findings per Domain
print('\n=== TABEL 4.4b: DISTRIBUSI FINDINGS PER DOMAIN ===')
compiled = [r for r in p2_results if r['compile_ok']]
header   = f'{"Domain":<14}' + ''.join(f'{n[:6]:>9}' for n in DETECTOR_NAMES) + f'{"TOTAL":>8}'
print(header)
print('─' * (14 + 9*len(DETECTOR_NAMES) + 8))
for domain in ['DeFi', 'NFT', 'Token', 'Governance', 'Utility']:
    dr = [r for r in compiled if r['domain'] == domain]
    vals = [sum(r.get(d, 0) for r in dr) for d in DETECTOR_NAMES]
    row  = f'{domain:<14}' + ''.join(f'{v:>9}' for v in vals) + f'{sum(vals):>8}'
    print(row)

# ── Tabel 4.4c: Top 10 Kontrak
print('\n=== TABEL 4.4c: TOP 10 KONTRAK DENGAN TEMUAN TERBANYAK ===')
print(f'{"No":>3} {"Nama":<35} {"Domain":<12} {"LOC":>6} {"Total":>7}')
print('─' * 70)
top10 = sorted(compiled,
               key=lambda r: sum(r.get(d, 0) for d in DETECTOR_NAMES),
               reverse=True)[:10]
for i, r in enumerate(top10, 1):
    total = sum(r.get(d, 0) for d in DETECTOR_NAMES)
    print(f'{i:>3}. {r["nama"]:<35} {r["domain"]:<12} {r["loc"]:>6} {total:>7}')

# ── Tabel 4.4d: Precision estimate (findings per LOC)
print('\n=== TABEL 4.4d: FINDINGS PER LOC (density) ===')
print(f'{"Complexity":<12} {"n":>4} {"Avg LOC":>9} {"Avg Findings":>14} {"Density":>9}')
print('─' * 55)
for cx in ['Simple', 'Medium', 'Complex']:
    group = [r for r in compiled if r['complexity'] == cx]
    if not group: continue
    avg_loc  = sum(r['loc'] for r in group) / len(group)
    avg_find = sum(sum(r.get(d, 0) for d in DETECTOR_NAMES) for r in group) / len(group)
    density  = avg_find / avg_loc * 100 if avg_loc else 0
    print(f'{cx:<12} {len(group):>4} {avg_loc:>9.0f} {avg_find:>14.1f} {density:>8.3f}%')

# ── Simpan tabel ke JSON
experiment_data = {
    'tabel_4_4a_gas_savings': bench_results,
    'tabel_4_4b_domain_dist': {
        domain: {
            d: sum(r.get(d, 0) for r in compiled if r['domain'] == domain)
            for d in DETECTOR_NAMES
        } for domain in ['DeFi', 'NFT', 'Token', 'Governance', 'Utility']
    },
    'tabel_4_4c_top10': [
        {'nama': r['nama'], 'domain': r['domain'],
         'total': sum(r.get(d, 0) for d in DETECTOR_NAMES)}
        for r in top10
    ],
    'total_findings': sum(r.get(d, 0) for r in compiled for d in DETECTOR_NAMES),
    'contracts_with_findings': sum(
        1 for r in compiled if sum(r.get(d, 0) for d in DETECTOR_NAMES) > 0
    ),
}
exp_path = RESULTS_DIR / 'pekan3_experiment_tables.json'
with open(exp_path, 'w') as f:
    json.dump(experiment_data, f, indent=2)
print(f'\n💾 Tabel eksperimen disimpan: {exp_path}')

=== TABEL 4.4a: GAS SAVINGS PER ANTI-PATTERN ===
 No Anti-Pattern                  Gas Boros  Gas Hemat   Selisih  % Hemat
───────────────────────────────────────────────────────────────────────────
  1. redundant_sload                  24,208     24,022       186     0.8%
  2. unoptimized_loop                 51,187     50,156     1,031     2.0%
  3. string_vs_bytes32                24,540     23,590       950     3.9%
  4. public_vs_external               52,544     49,871     2,673     5.1%
  5. unchecked_arithmetic             59,105     47,060    12,045    20.4%
  6. dead_code                       123,985    123,985         0     0.0%

=== TABEL 4.4b: DISTRIBUSI FINDINGS PER DOMAIN ===
Domain           redund   unopti   string   public   unchec   dead_c   TOTAL
────────────────────────────────────────────────────────────────────────────
DeFi                151        2       24       56        0       29     262
NFT                 208        0      112      183        0       36

---
## ✅ Sel 8 — Checklist Akhir Pekan 3

In [8]:
checks = [
    ('src/gas_estimator.py ada',
        (ROOT / 'src' / 'gas_estimator.py').exists()),
    ('src/refactorer.py ada',
        (ROOT / 'src' / 'refactorer.py').exists()),
    ('GasBenchmark.sol ditulis ke hardhat_project',
        (HARDHAT_DIR / 'contracts' / 'GasBenchmark.sol').exists()),
    ('Benchmark berhasil dijalankan (6 pattern)',
        len(bench_results) == 6),
    ('5+ dari 6 benchmark ada penghematan gas',
        sum(1 for r in bench_results if r['diff_gas'] > 0) >= 5),
    ('Refactorer berjalan tanpa error',
        True),  # jika Sel 3 selesai
    ('Refactored code masih compile',
        ok_refc),
    ('Tabel eksperimen disimpan',
        (RESULTS_DIR / 'pekan3_experiment_tables.json').exists()),
    ('Slither dijalankan (atau tersedia)',
        slither_ok or bool(slither_results)),
]

print('=' * 52)
print('  CHECKLIST AKHIR PEKAN 3')
print('=' * 52)

semua_ok = True
for label, ok in checks:
    icon = '✅' if ok else '❌'
    if not ok: semua_ok = False
    print(f'  {icon} {label}')

print('─' * 52)
print(f'  Benchmark patterns   : {len(bench_results)}/6')
avg_save = sum(r["pct_save"] for r in bench_results) / len(bench_results)
print(f'  Avg gas savings      : {avg_save:.1f}%')
print(f'  Refactoring patches  : {sum(summary.values())} patch')
print(f'  Slither comparison   : {"aktif" if slither_ok else "skip (belum install)"}')
print('=' * 52)
print('  🎉 Siap masuk Pekan 4!' if semua_ok
      else '  ⚠️  Cek item ❌ di atas')
print('=' * 52)


  CHECKLIST AKHIR PEKAN 3
  ✅ src/gas_estimator.py ada
  ✅ src/refactorer.py ada
  ✅ GasBenchmark.sol ditulis ke hardhat_project
  ✅ Benchmark berhasil dijalankan (6 pattern)
  ✅ 5+ dari 6 benchmark ada penghematan gas
  ✅ Refactorer berjalan tanpa error
  ✅ Refactored code masih compile
  ✅ Tabel eksperimen disimpan
  ✅ Slither dijalankan (atau tersedia)
────────────────────────────────────────────────────
  Benchmark patterns   : 6/6
  Avg gas savings      : 5.4%
  Refactoring patches  : 26 patch
  Slither comparison   : aktif
  🎉 Siap masuk Pekan 4!
